# 04 — Clustering Analysis

Group banks by their intraday liquidity behaviour using K-Means clustering.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import load_raw_data, preprocess
from src.feature_engineering import build_features, get_feature_matrix
from src.clustering import LiquidityClustering

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Prepare bank-level feature profiles

In [ ]:
df_raw = load_raw_data('../data/raw/synthetic_intraday_payments_500k.csv')
df = preprocess(df_raw)

# Aggregate to bank-level daily features
bank_profiles = (
    df.groupby('bank_id')
    .agg(
        mean_amount=('amount', 'mean'),
        std_amount=('amount', 'std'),
        total_volume=('amount', 'sum'),
        txn_count=('transaction_id', 'count'),
        mean_hour=('hour', 'mean'),
        credit_ratio=('transaction_type', lambda x: (x == 'CREDIT').mean()),
        net_position=('signed_amount', 'sum'),
    )
    .reset_index()
)
print(bank_profiles)


## 2. Find optimal number of clusters (elbow method)

In [ ]:
feature_cols = ['mean_amount', 'std_amount', 'total_volume', 'txn_count',
                'mean_hour', 'credit_ratio', 'net_position']
X = bank_profiles[feature_cols].fillna(0)

elbow_df = LiquidityClustering.find_optimal_k(X, k_range=range(2, min(len(X), 10)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(elbow_df['k'], elbow_df['inertia'], marker='o')
ax1.set_title('Elbow Curve')
ax1.set_xlabel('k')
ax1.set_ylabel('Inertia')

ax2.plot(elbow_df['k'], elbow_df['silhouette_score'], marker='o', color='green')
ax2.set_title('Silhouette Score')
ax2.set_xlabel('k')
ax2.set_ylabel('Score')

plt.tight_layout()
plt.show()
print(elbow_df)


## 3. Fit clustering model

In [ ]:
n_clusters = 4
clusterer = LiquidityClustering(n_clusters=n_clusters, method='kmeans')
bank_profiles['cluster'] = clusterer.fit_predict(X)
print(bank_profiles[['bank_id', 'cluster']].sort_values('cluster'))


## 4. Cluster evaluation

In [ ]:
metrics = clusterer.evaluate(X)
print('Clustering metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')


## 5. Visualise clusters (PCA)

In [ ]:
labels = bank_profiles['cluster'].values
clusterer.plot_clusters(
    X,
    labels,
    save_path='../figures/clustering.png',
    title='Bank Liquidity Clusters (PCA projection)',
)
plt.show()


## 6. Cluster profiles

In [ ]:
profiles = clusterer.cluster_profiles(bank_profiles, feature_cols)
print(profiles.round(2))


## 7. Save clustering model

In [ ]:
clusterer.save('../models/cluster_model.pkl')
print('Cluster model saved.')
